# FooDB v1.0 cleanup

Cleans the [FooDB](https://foodb.ca/) v1.0 database (release 2020-04-07) so that it contains individual food constituents with standardized identifiers. FooDB is the world's largest and most comprehensive resource on food constituents, chemistry and biology. It covers ca. 70,000 compounds found in ca. 800 food stuffs, linked through a content table that reports compound concentrations in individual foods.

**Workflow**

| Step | What happens |
| --- | --- |
| 1 | Load the Content CSV |
| 2 | Enrich the raw Compound table with PubChem data (names, IUPAC names, InChIKeys); resumable, cached |
| 3 | Load the Food and enriched Compound tables |
| 4 | Clean the Food table and create a harmonized food group |
| 5 | Clean the Compound table (drop columns, fix broken names, remove missing InChIKeys) |
| 6 | Remove unwanted compounds (by InChIKey and keywords) |
| 7 | Clean the Content table (rename columns, drop columns, remove nutrients and summary entries) |
| 8 | Populate missing values in the Content table |
| 9 | Remove Content entries (missing names, unknown compounds, empty or zero content) |
| 10 | Show stats for the filtered tables |
| 11 | Save the filtered tables |
| 12 | Merge the filtered tables and save the result |
| 13 | Export per-compound statistics |

**Settings**

All user-editable settings are in the configuration cell:

- `FOODB_CSV_DIR`: folder with the raw FooDB CSV files (`Content.csv`, `Food.csv`, `Compound.csv`; download from https://foodb.ca/downloads).
- `OUTPUT_DIR` / `CACHE_DIR`: output (`foodb_cleaned`) and cache (`cache`) folders, created next to the notebook.
- `LOOKUP_LIMIT`: set to e.g. 50 for a quick test run of the PubChem lookups; `None` runs all compounds.
- `MAX_REQ_PER_SEC` and `N_WORKERS`: PubChem request rate (PubChem requests <= 5 requests/second) and number of parallel workers.
- `CHUNK`: number of compounds per cache write (and InChIKeys per PubChem request); smaller chunks give a more responsive ETA.

**Requirements:** Python 3 with `pandas`, `chardet` and `requests` (the latter used by the shared PubChem client), the shared `inquire_common` folder (utilities, PubChem client and filter lists) in the parent folder of this notebook, and internet access for the PubChem step.

**Output files** (written to `foodb_cleaned/`)

| File | Description |
| --- | --- |
| `Compound_FDB_PubChemNames.csv` | Raw Compound table with PubChem names and InChIKeys |
| `Food_Fltrd.csv` | Cleaned Food table |
| `Compound_Fltrd.csv` | Cleaned and filtered Compound table |
| `Content_Fltrd.csv` | Cleaned and filtered Content table |
| `FooDB_Merged.csv` | Merged Content, Food and Compound tables (tab-separated) |
| `FooDB_03_Compound_Food_Composition.csv` | One row per compound with row counts and number of distinct foods and food groups |

In [1]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Environment reset | 2026-11-18 12:34:55



In [2]:
# Configuration & imports
import os
import pandas as pd
from datetime import datetime
import re
import time
import chardet
from concurrent.futures import ThreadPoolExecutor

# Shared helpers/filters (common lives in the parent folder of this notebook)
import sys
sys.path.insert(0, os.path.abspath(".."))
from common.utils import print_overwrite, df_stats
from common.compound_filters import apply_standard_filters
from common.pubchem import PubChemClient, run_cached_lookup
from common.cleaning import add_uid, migrate_cache_key, read_cache

# FooDB cleanup rule tables (rules/*.csv) and the shared loader (inquire_common/rules.py)
from common.rules import RuleSet
rules = RuleSet("rules")

# ---------------------------------------------------------------------------
# CONFIGURATION (all user-editable settings live here)
# ---------------------------------------------------------------------------
# Folder holding the raw FooDB CSV files
FOODB_CSV_DIR = r"../original_databases/foodb_2020_04_07_csv"

# Raw input files
CSV_PATH = os.path.join(FOODB_CSV_DIR, "Content.csv")
RAW_COMPOUND_CSV = os.path.join(FOODB_CSV_DIR, "Compound.csv")
Food_CSV_PATH = os.path.join(FOODB_CSV_DIR, "Food.csv")

# Output & cache folders (subfolders of foodb_v1.0_cleanup-main, i.e. the notebook folder)
BASE_DIR = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "foodb_cleaned")
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Cache file (resumable PubChem lookups, one row per unique identifier set (uid))
PUBCHEM_CACHE_CSV = os.path.join(CACHE_DIR, "PubChem_lookup_cache.csv")

# Output files
ENRICHED_COMPOUND_CSV = os.path.join(OUTPUT_DIR, "Compound_FDB_PubChemNames.csv")  # used by the rest of the notebook
Compound_CSV_PATH = ENRICHED_COMPOUND_CSV
FOOD_FLTRD_CSV = os.path.join(OUTPUT_DIR, "Food_Fltrd.csv")
COMPOUND_FLTRD_CSV = os.path.join(OUTPUT_DIR, "Compound_Fltrd.csv")
CONTENT_FLTRD_CSV = os.path.join(OUTPUT_DIR, "Content_Fltrd.csv")
MERGED_CSV = os.path.join(OUTPUT_DIR, "FooDB_Merged.csv")
STATS_CSV = os.path.join(OUTPUT_DIR, "FooDB_03_Compound_Food_Composition.csv")  # per-compound statistics (step 13)

# PubChem enrichment settings
LOOKUP_LIMIT = None      # set to e.g. 50 for a quick test run; None = all compounds
MAX_REQ_PER_SEC = 4      # PubChem asks for <= 5 requests/second. You may be blocked when increasing the rate.
N_WORKERS = 4
CHUNK = 200              # compounds per cache write (= InChIKeys per PubChem POST request); smaller chunks give a more responsive ETA

print(f"Configuration & imports done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Configuration & imports done | 2026-11-18 12:34:55



In [3]:
# Helper to remove previous progress message
# print_overwrite is imported from inquire_common.utils in the configuration cell

print(f"Helper to remove previous progress message created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Helper to remove previous progress message created | 2026-11-18 09:22:59



## Step 1: Load the Content CSV

Paths and settings are defined in the configuration cell above. Confirm that the Content CSV exists before reading it.

In [4]:
# Check Content CSV path (set in the configuration cell)
assert os.path.exists(CSV_PATH), f"Content CSV file not found: {CSV_PATH}"
print(f"Content CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Content CSV path is valid | 2026-11-18 09:22:59



Read the Content CSV into `content_df`.

In [5]:
# Reading CSV file
print(f"Reading Content CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Read entire CSV
content_df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Content CSV shape: \n{content_df.shape[0]:,} rows × {content_df.shape[1]} columns\n")
print(f"Content CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Reading Content CSV with pandas | 2026-11-18 09:22:59
Content CSV shape: 
5,145,532 rows × 26 columns

Content CSV loaded | 2026-11-18 09:23:06



## Step 2: Enrich the raw Compound table with PubChem data

Built in; no separate tool is needed. First confirm that the raw `Compound.csv` exists (path set in the configuration cell).

In [6]:
# Raw FooDB Compound.csv -> notebook schema + PubChem enrichment: path check (settings are in the configuration cell)
assert os.path.exists(RAW_COMPOUND_CSV), f"Raw Compound CSV not found: {RAW_COMPOUND_CSV}"
print(f"Raw Compound CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Raw Compound CSV path is valid | 2026-11-18 09:23:06



The next cell enriches the raw Compound table:

1. **Relabel columns by position:** the 2020-04-07 `Compound.csv` header does not match the data order (e.g., the column labelled `moldb_inchikey` holds InChI strings). The InChIKey and InChI columns are sanity-checked afterwards.
2. **Rename columns** to follow other similar databases, e.g., `id` to `cpd_id`, `name` to `orig_name` and `moldb_iupac` to `iupac_name`.
3. **Retrieve PubChem data** (names, IUPAC names, InChIKeys) via the PubChem PUG REST API: batched InChIKey lookups first, then a name-based fallback for misses.
4. **Rate limiting and caching:** requests are rate limited (`MAX_REQ_PER_SEC`, adapting to the `X-Throttling-Control` header), run in parallel (`N_WORKERS`) and cached in `cache/PubChem_lookup_cache.csv`, so the step is resumable and safe to interrupt and re-run.
5. **Save** the enriched table as `foodb_cleaned/Compound_FDB_PubChemNames.csv`.

In [7]:
# Rename raw FooDB Compound columns & retrieve PubChem names/InChIKeys (batched PUG REST via POST)

raw_cpd = pd.read_csv(RAW_COMPOUND_CSV, low_memory=False)

# 0. the header row of the 2020-04-07 Compound.csv does not match the data order (e.g. the column labelled
#    'moldb_inchikey' holds InChI strings), so relabel the columns by position and sanity-check the result
raw_cpd.columns = [
    "id", "public_id", "name", "state", "annotation_quality", "description", "cas",
    "moldb_smiles", "moldb_inchi", "moldb_mono_mass", "moldb_inchikey", "moldb_iupac",
    "kingdom", "superklass", "klass", "subklass",
]
_ik = raw_cpd["moldb_inchikey"].dropna().astype(str)
assert _ik.str.fullmatch(r"[A-Z]{14}-[A-Z]{10}-[A-Z]").all(), "Compound.csv column layout differs from expected"
assert raw_cpd["moldb_inchi"].dropna().astype(str).str.startswith("InChI=").all(), "Compound.csv column layout differs from expected"

# 1. rename to the schema used in the rest of the notebook
raw_cpd = raw_cpd.rename(columns={
    "id": "cpd_id",
    "public_id": "public_cpd_id",
    "name": "orig_name",
    "moldb_iupac": "iupac_name",
    "moldb_inchikey": "inchikey",
})

# 2. PubChem client (rate limiter + request helpers live in inquire_common/pubchem.py)
client = PubChemClient(max_req_per_sec=MAX_REQ_PER_SEC)

def lookup_compounds(rows):
    """rows: list of (uid, name, inchikey). Batched InChIKey lookup first, then name fallback for misses.
    Compounds hitting a transient failure are omitted so a re-run retries them."""
    hits = {}
    keys = sorted({ik.strip() for _, _, ik in rows if isinstance(ik, str) and ik.strip()})
    failed_keys = set()
    for i in range(0, len(keys), CHUNK):
        batch = keys[i:i + CHUNK]
        try:
            hits.update(client.batch_by_inchikey(batch))
        except RuntimeError:
            failed_keys.update(batch)

    def _fallback(row):
        uid, name, ik = row
        ik = ik.strip() if isinstance(ik, str) else None
        if ik in failed_keys:
            return None                                # transient failure: leave uncached
        hit = hits.get(ik) if ik else None
        if hit is None and isinstance(name, str) and name.strip():
            try:
                hit = client.name_lookup(name.strip())
                hit = hit[1:] if hit else None   # drop the CID: (Title, IUPACName, InChIKey)
            except RuntimeError:
                return None
        t, i, k = hit if hit else (None, None, None)
        return {"uid": uid, "Name_PubChem": t, "IUPACName_PubChem": i, "InChIKey_PubChem": k}

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        return [r for r in ex.map(_fallback, rows) if r is not None]

# 3. resumable lookup (cache keyed by uid, like the other source databases)
UNIQUE_KEY_COLS = ["cpd_id", "orig_name", "inchikey"]
df_unique = add_uid(raw_cpd, UNIQUE_KEY_COLS)
migrate_cache_key(PUBCHEM_CACHE_CSV, df_unique, "cpd_id")  # converts a cache from before the uid key, if present

cache_cols = ["uid", "Name_PubChem", "IUPACName_PubChem", "InChIKey_PubChem"]
run_cached_lookup(
    df_unique, ["orig_name", "inchikey"], lookup_compounds,
    PUBCHEM_CACHE_CSV, cache_cols, client, chunk=CHUNK, limit=LOOKUP_LIMIT,
)

# 4. merge cache into the compound table & save
df_unique_pc = df_unique.merge(read_cache(PUBCHEM_CACHE_CSV, cache_cols), on="uid", how="left", validate="1:1")
enriched = raw_cpd.merge(df_unique_pc.drop(columns="uid"), on=UNIQUE_KEY_COLS, how="left", validate="m:1")
assert len(enriched) == len(raw_cpd), "Row count changed during the PubChem merge"
enriched.to_csv(ENRICHED_COMPOUND_CSV, index=False, encoding="utf-8")

print(f"\n\nEnriched Compound table: {enriched.shape[0]:,} rows x {enriched.shape[1]} columns")
print(f"Rows with PubChem name: {enriched['Name_PubChem'].notna().sum():,}")
print(f"Saved to {ENRICHED_COMPOUND_CSV} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


70,477 of 70,477 compounds already cached, 0 to look up | 2026-11-18 09:23:07
No up-front time estimate: an Estimated Time of Arrival based on measured throughput is shown while running.
It is safe to interrupt and re-run.



Cached compounds: 70,477 of 70,477
Resolved to an InChIKey: 65,423

PubChem retrieval done | 2026-11-18 09:23:07



Enriched Compound table: 70,477 rows x 19 columns
Rows with PubChem name: 65,272
Saved to ./foodb_cleaned/Compound_FDB_PubChemNames.csv | 2026-11-18 09:23:08



## Step 3: Load the Food and enriched Compound tables

Confirm that the Food CSV and the enriched Compound CSV from step 2 exist, then read them.

In [8]:
# Check Food & Compound CSV paths (set in the configuration cell)
assert os.path.exists(Food_CSV_PATH), f"Food CSV file not found: {Food_CSV_PATH}"
print(f"Food CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

assert os.path.exists(Compound_CSV_PATH), f"Compound CSV file not found: {Compound_CSV_PATH}"
print(f"Compound CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Food CSV path is valid | 2026-11-18 09:23:08

Compound CSV path is valid | 2026-11-18 09:23:08



Read the enriched Compound CSV into `cpd_df`. The encoding is detected (with `chardet`), with fallback encodings if needed.

In [9]:
# Reading Compound CSV file
print(f"Reading Compound CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Check encoding
with open(Compound_CSV_PATH, "rb") as f:
    result = chardet.detect(f.read(100000))
print(f"Compound CSV file encoding: \n{result}\n")

# Read entire CSV
cpd_df = pd.read_csv(Compound_CSV_PATH, encoding=result["encoding"], low_memory=False)

print(f"Compound CSV shape: \n{cpd_df.shape[0]:,} rows × {cpd_df.shape[1]} columns\n")
print(f"Compound CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Reading Compound CSV with pandas | 2026-11-18 09:23:08

Compound CSV file encoding: 
{'encoding': 'utf-8', 'confidence': 0.8015960000000001, 'language': 'cy', 'mime_type': 'text/plain'}

Compound CSV shape: 
70,477 rows × 19 columns

Compound CSV loaded | 2026-11-18 09:23:09



Read the Food CSV into `food_df`.

In [10]:
# Reading Food CSV file
print(f"Reading Food CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check encoding
with open(Food_CSV_PATH, "rb") as f:
    rawdata = f.read(100000)
    result = chardet.detect(rawdata)
print(f"Food CSV file encoding: \n{result}\n")

# Try detected encoding first
try:
    food_df = pd.read_csv(Food_CSV_PATH, encoding=result["encoding"], low_memory=False)
except UnicodeDecodeError:
    print(f"Detected encoding failed. Trying fallback encodings...\n")
    
    fallback_encodings = ["latin1", "cp1252", "ISO-8859-1"    ]

    for enc in fallback_encodings:
        try:
            print(f"Trying encoding: {enc}")

            food_df = pd.read_csv(Food_CSV_PATH, encoding=enc, low_memory=False)

            print(f"Successfully loaded using: {enc}\n")
            break
        except UnicodeDecodeError:
            print(f"Failed with: {enc}")

print(f"Food CSV shape: \n{food_df.shape[0]:,} rows × {food_df.shape[1]} columns\n")
print(f"Food CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Reading Food CSV with pandas | 2026-11-18 09:23:09
Food CSV file encoding: 
{'encoding': 'utf-8', 'confidence': 0.8025992000000001, 'language': 'en', 'mime_type': 'text/plain'}

Food CSV shape: 
992 rows × 23 columns

Food CSV loaded | 2026-11-18 09:23:09



## Step 4: Clean the Food table

- Renames columns to follow other similar databases, e.g., `id` to `food_id`, `name` to `food_name` and `public_id` to `public_food_id`.
- Drops unnecessary columns (e.g., picture data, timestamps, creator/updater IDs, export flags).

In [11]:
# food_df cleanup
print(f"Renaming & dropping food_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename columns
food_df_rnmd = food_df.rename(columns={
    'id': 'food_id',
    'name': 'food_name',
    'name_scientific': 'food_name_scientific',
    'public_id': 'public_food_id'
})

# drop unnecessary columns
food_df_drp = food_df_rnmd.drop(columns=[
    'wikipedia_id',
    'picture_file_name',
    'picture_content_type',
    'picture_file_size',
    'picture_updated_at',
    'created_at',
    'updated_at',
    'creator_id',
    'updater_id',
    'export_to_afcdb',
    'category',
    'export_to_foodb'
], errors='ignore')

print(f"Initial columns: \n{food_df.columns.tolist()}\n")
print(f"Remaining columns: \n{food_df_drp.columns.tolist()}\n")
print(f"Done renaming & dropping food_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Renaming & dropping food_df columns | 2026-11-18 09:23:09

Initial columns: 
['id', 'name', 'name_scientific', 'description', 'itis_id', 'wikipedia_id', 'picture_file_name', 'picture_content_type', 'picture_file_size', 'picture_updated_at', 'legacy_id', 'food_group', 'food_subgroup', 'food_type', 'created_at', 'updated_at', 'creator_id', 'updater_id', 'export_to_afcdb', 'category', 'ncbi_taxonomy_id', 'export_to_foodb', 'public_id']

Remaining columns: 
['food_id', 'food_name', 'food_name_scientific', 'description', 'itis_id', 'legacy_id', 'food_group', 'food_subgroup', 'food_type', 'ncbi_taxonomy_id', 'public_food_id']

Done renaming & dropping food_df columns | 2026-11-18 09:23:09



Create and populate a harmonized food group column (`food_group_harmonized`: `food_group | food_subgroup | food_type`), similar to the CPDat product use category.

In [12]:
# create harmonized food group (similar to CPDat product use category)
print(f"Creating & populating harmonized food group (similar to CPDat product use category) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# copy food_df_drp
food_df_harm = food_df_drp.copy()

HARMONIZED_SEPARATOR = " | "

# create column if not present
if "food_group_harmonized" not in food_df_harm.columns:
    food_df_harm["food_group_harmonized"] = None

# Move food_group_harmonized after food_type
cols = list(food_df_harm.columns)
if "food_type" in cols and "food_group_harmonized" in cols:
    cols.insert(
        cols.index("food_type") + 1,
        cols.pop(cols.index("food_group_harmonized"))
    )
    food_df_harm = food_df_harm[cols]

print(f"Created 'food_group_harmonized' col | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# helper: replace missing with literal "NaN"
def harmonized_value(x):
    return x if isinstance(x, str) and x.strip() else "NaN"

# populate puc_harmonized
food_df_harm["food_group_harmonized"] = food_df_harm.apply(
    lambda row: HARMONIZED_SEPARATOR.join([
        harmonized_value(row.get("food_group")),
        harmonized_value(row.get("food_subgroup")),
        harmonized_value(row.get("food_type")),
    ]),
    axis=1
)

print(f"Created & populated harmonized food group (similar to CPDat product use category) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Creating & populating harmonized food group (similar to CPDat product use category) | 2026-11-18 09:23:09

Created 'food_group_harmonized' col | 2026-11-18 09:23:09

Created & populated harmonized food group (similar to CPDat product use category) | 2026-11-18 09:23:09



## Step 5: Clean the Compound table

- Drops unnecessary columns (e.g., `state`, `annotation_quality`, `kingdom`).
- Detects broken compound names (mojibake) and replaces them with the existing IUPAC name, the PubChem IUPAC name or the PubChem name.

In [13]:
# cpd_df cleanup
print(f"Renaming columns & replacing borken names with IUPAC names in cpd_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# drop unnecessary columns & rename PubChem columns (needed below for the name replacement)
cpd_df_drp = cpd_df.drop(columns=[
    'state',
    'annotation_quality',
    'kingdom'
], errors='ignore').rename(columns={
    'Name_PubChem': 'name',
    'IUPACName_PubChem': 'iupac_name_pubchem'
})

# detect mojibake / broken text in orig_name & replace with iupac_name
def has_mojibake(text):
    if pd.isna(text):
        return False

    text = str(text)

    # common mojibake fragments
    mojibake_patterns = [
        r'Ã.',
        r'Â.',
        r'â..',
        r'�',
        r'¤',
        r'�'
    ]

    for pattern in mojibake_patterns:
        if re.search(pattern, text):
            return True

    # excessive non-ascii control-like artifacts
    weird_ratio = sum(ord(c) > 127 for c in text) / max(len(text), 1)

    if weird_ratio > 0.3:
        return True

    return False

cpd_df_moji = cpd_df_drp.copy()

# replace corrupted orig_name with iupac_name
mask = cpd_df_drp['orig_name'].apply(has_mojibake)

# print replacements before overwriting
print(f"{len(cpd_df_drp.loc[mask, 'orig_name'])} broken compound names found \nReplacing with existing IUPAC names & IUPAC names from PubChem")

print("\nDetected mojibake replacements:\n")
for old_val, new_val in zip(
    cpd_df_drp.loc[mask, 'orig_name'],
    cpd_df_drp.loc[mask, 'iupac_name']
):
    print(f"{old_val} >>> {new_val}")

# replace corrupted orig_name with iupac_name
cpd_df_moji.loc[mask, 'orig_name'] = (
    cpd_df_drp.loc[mask, ['iupac_name', 'iupac_name_pubchem', 'name']]
    .replace(r'^\s*$', pd.NA, regex=True)
    .bfill(axis=1)
    .iloc[:, 0]
)

print(f"\nInitial columns: \n{cpd_df.columns.tolist()}\n")
print(f"Remaining columns: \n{cpd_df_moji.columns.tolist()}\n")
print(f"Done renaming columns & replacing borken names with IUPAC names in cpd_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Renaming columns & replacing borken names with IUPAC names in cpd_df | 2026-11-18 09:23:09

0 broken compound names found 
Replacing with existing IUPAC names & IUPAC names from PubChem

Detected mojibake replacements:


Initial columns: 
['cpd_id', 'public_cpd_id', 'orig_name', 'state', 'annotation_quality', 'description', 'cas', 'moldb_smiles', 'moldb_inchi', 'moldb_mono_mass', 'inchikey', 'iupac_name', 'kingdom', 'superklass', 'klass', 'subklass', 'Name_PubChem', 'IUPACName_PubChem', 'InChIKey_PubChem']

Remaining columns: 
['cpd_id', 'public_cpd_id', 'orig_name', 'description', 'cas', 'moldb_smiles', 'moldb_inchi', 'moldb_mono_mass', 'inchikey', 'iupac_name', 'superklass', 'klass', 'subklass', 'name', 'iupac_name_pubchem', 'InChIKey_PubChem']

Done renaming columns & replacing borken names with IUPAC names in cpd_df | 2026-11-18 09:23:10



Remove compounds with no InChIKey value, then reorder the (already renamed) PubChem columns (`name`, `iupac_name_pubchem`).

In [14]:
# cpd_df_moji cleanup
print(f"Removing rows with missing InChIKeys, renaming & resorting columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

cpd_df_clean = cpd_df_moji.copy()

print(f"Initial row count: {len(cpd_df_clean):,}\n")

# remove rows with missing inchikey
col_name = 'inchikey'

if col_name in cpd_df_clean.columns:
    rows_before = len(cpd_df_clean)

    # detect missing / empty values
    mask_remove = (cpd_df_clean[col_name].isna() |
        (cpd_df_clean[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()

    print(f"Rows with missing inchikey: {(n_removed):,}")

    cpd_df_clean = cpd_df_clean.loc[~mask_remove].copy()
    rows_after = len(cpd_df_clean)

    print(f"Rows before: {(rows_before):,}")
    print(f"Rows after: {(rows_after):,}")
else:
    print(f"Column '{col_name}' not found\n")


# reorder PubChem columns (already renamed to name / iupac_name_pubchem in the previous cell)
cols_to_place = ['orig_name', 'name', 'iupac_name_pubchem']

if all(col in cpd_df_clean.columns for col in cols_to_place):
    cols = cpd_df_clean.columns.tolist()

    # remove columns to reposition
    cols.remove('name')
    cols.remove('iupac_name_pubchem')

    # insert name after orig_name
    idx_orig = cols.index('orig_name')
    cols.insert(idx_orig + 1, 'name')

    # insert iupac_name_pubchem after name
    idx_feat = cols.index('name')
    cols.insert(idx_feat + 1, 'iupac_name_pubchem')

    # reorder dataframe
    cpd_df_clean = cpd_df_clean[cols]

    print(f"PubChem columns reordered successfully\n")
else:
    print(f"Required columns for reordering not found\n")

print(f"Final row count: {len(cpd_df_clean):,}\n")
print(f"Final columns:")
print(cpd_df_clean.columns.tolist())

print(f"\nDone removing rows with missing InChIKeys, renaming & resorting columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Removing rows with missing InChIKeys, renaming & resorting columns | 2026-11-18 09:23:10

Initial row count: 70,477

Rows with missing inchikey: 62
Rows before: 70,477
Rows after: 70,415
PubChem columns reordered successfully

Final row count: 70,415

Final columns:
['cpd_id', 'public_cpd_id', 'orig_name', 'name', 'iupac_name_pubchem', 'description', 'cas', 'moldb_smiles', 'moldb_inchi', 'moldb_mono_mass', 'inchikey', 'iupac_name', 'superklass', 'klass', 'subklass', 'InChIKey_PubChem']

Done removing rows with missing InChIKeys, renaming & resorting columns | 2026-11-18 09:23:10



## Step 6: Remove compounds

Removes compounds:

1. With specific InChIKey values (e.g., mixtures, inorganic compounds and inorganic mixtures, isotopically labeled compounds, ions).
2. With unwanted keywords in the name (e.g., mixtures, minerals, metals, inorganic compounds and inorganic mixtures, extracts), except for protected compound names.

In [15]:
# Remove chemicals by InChIKey, names, and keywords from cpd_df_clean
print(f"Removing features by InChIKey, names & keywords from cpd_df_clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Lists live in inquire_common/filters: drop_inchikeys.txt + drop_inchikeys_foodb.txt, keywords_common.txt +
# keywords_foodb.txt, keyword_exceptions.txt. FooDB has no-InChIKey rows removed earlier, and matches stems
# (anchor_end=False) ON PURPOSE: FooDB has many salts, elements and derivatives (magnesium, manganese,
# molybdopterin, selenomethionine, ...) that the stems in keywords_foodb.txt are meant to catch, so stem
# matching removes more names than whole-word matching. The SPECIATE/SPECIEUROPE/CPDat notebooks use True.
initial_n = len(cpd_df_clean)
cpd_df_fltrd, dropped_df, dropped_chems = apply_standard_filters(
    cpd_df_clean, source="foodb",
    anchor_end=False, drop_missing_inchikey=False, inchikey_sources=["foodb"], label="cpd_df_clean",
)

# Log what stem matching adds over whole-word matching (silent re-run with anchor_end=True)
n_whole_word = len(apply_standard_filters(
    cpd_df_clean, source="foodb", anchor_end=True, drop_missing_inchikey=False,
    inchikey_sources=["foodb"], label="cpd_df_clean", verbose=False,
)[0])
print(f"Rows kept with anchor_end=True (whole words): {n_whole_word:,}; anchor_end=False (stems): {len(cpd_df_fltrd):,}; "
      f"stem matching removes {n_whole_word - len(cpd_df_fltrd):,} extra rows\n")

print(f"Done removing features by InChIKey, names & keywords from cpd_df_clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Removing features by InChIKey, names & keywords from cpd_df_clean | 2026-11-18 09:23:10

Number of entries in cpd_df_clean before InChIKey & keyword filtration: 
70,415

Dropped compounds by InChIKey:
ABLZXFCXXLZCGV-UHFFFAOYSA-L >>> CID 406
AVXURJPOCDRRFD-UHFFFAOYSA-N >>> Hydroxylamine
AXZWODMDQAVCJE-UHFFFAOYSA-L >>> Stannous Chloride
AZDRQVAHHNSJOQ-UHFFFAOYSA-N >>> Aluminum hydride
BDAGIHXWWSANSR-UHFFFAOYSA-M >>> Formate
BFNBIHQBYMNNAN-UHFFFAOYSA-N >>> Ammonium sulfate, ACS reagent
BQPIGGFYSBELGY-UHFFFAOYSA-N >>> Mercuric cation
BVKZGUZCCUSVTD-UHFFFAOYSA-L >>> Carbonate Ion
BVKZGUZCCUSVTD-UHFFFAOYSA-N >>> Hydrogen carbonate;hydron
BWGNESOTFCXPMA-UHFFFAOYSA-N >>> Hydrogen disulfide
CMFNMSMUKZHDEY-UHFFFAOYSA-M >>> Peroxynitrite radical
CPELXLSAUQHCOX-UHFFFAOYSA-M >>> Bromide ion
CWYNVVGOOAEACU-UHFFFAOYSA-N >>> Ferrous cation
DHCDFWKWKRSZHF-UHFFFAOYSA-L >>> Thiosulfate ion
FKNQFGJONOIPTF-UHFFFAOYSA-N >>> Sodium Ion
FPAFDBFIGPHWGO-UHFFFAOYSA-N >>> Dioxosilane;oxomagnesium;hydrate
GDTBXPJZ

## Step 7: Clean the Content table

- Renames columns to follow other similar databases, e.g., `id` to `cont_id`, `source_id` to `cpd_id` and `orig_source_name` to `orig_name`.
- Drops unnecessary columns (e.g., citations, timestamps, creator/updater IDs, methods, units expressions, standard content, preparation type, export flag).

In [16]:
# content_df cleanup
print(f"Renaming & dropping content_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename columns
content_df_rnmd = content_df.rename(columns={
    'id': 'cont_id',
    'source_id': 'cpd_id',
    'orig_source_name': 'orig_name'
})

print(f"content_df columns renamed\n")

# drop unnecessary columns
content_df_drp = content_df_rnmd.drop(columns=[
    'orig_citation',
    'citation',
    'citation_type',
    'creator_id',
    'updater_id',
    'created_at',
    'updated_at',
    'orig_method',
    'orig_unit_expression',
    'standard_content',
    'preparation_type',
    'export'
], errors='ignore')

print(f"content_df unnecessary columns dropped\n")
print(f"Initial columns: \n{content_df.columns.tolist()}\n")
print(f"Remaining columns: \n{content_df_drp.columns.tolist()}\n")
print(f"Done renaming & dropping content_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Renaming & dropping content_df columns | 2026-11-18 09:23:52

content_df columns renamed

content_df unnecessary columns dropped

Initial columns: 
['id', 'source_id', 'source_type', 'food_id', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'orig_source_name', 'orig_content', 'orig_min', 'orig_max', 'orig_unit', 'orig_citation', 'citation', 'citation_type', 'creator_id', 'updater_id', 'created_at', 'updated_at', 'orig_method', 'orig_unit_expression', 'standard_content', 'preparation_type', 'export']

Remaining columns: 
['cont_id', 'cpd_id', 'source_type', 'food_id', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'orig_name', 'orig_content', 'orig_min', 'orig_max', 'orig_unit']

Done renaming & dropping content_df columns | 2026-11-18 09:23:52



Remove nutrients (`source_type == 'Nutrient'`) and summary entries (e.g., 'Carbohydrates, total', 'Fat, total (Lipids)').

In [17]:
# content_df_drp cleanup
print(f"Dropping rows where source_type == 'Nutrient' & orig_name contains 'total' | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_nutr = content_df_drp.copy()

print(f"Initial row count: {len(content_df_nutr):,}\n")

# remove rows where source_type == 'Nutrient'
col_name = 'source_type'
target_value = 'Nutrient'

if col_name in content_df_nutr.columns:
    rows_before = len(content_df_nutr)

    # ensure string-safe comparison
    mask_remove = (
        content_df_nutr[col_name]
        .astype(str)
        .str.strip()
        .eq(target_value)
    )

    n_removed = mask_remove.sum()
    content_df_nutr = content_df_nutr.loc[~mask_remove].copy()
    content_df_nutr = content_df_nutr.drop(columns=['source_type'], errors='ignore')
    rows_after = len(content_df_nutr)

    print(f"Removed rows where {col_name} == '{target_value}': {(n_removed):,}")
    print(f"Rows before: {(rows_before):,}")
    print(f"Rows after: {(rows_after):,}\n")
else:
    print(f"\nColumn '{col_name}' not found. Skipping cleanup.\n")


# remove rows where orig_name contains 'total'
content_df_ttl = content_df_nutr.copy()

col_name_2 = 'orig_name'
target_patterns = rules.load_list("summary_name_patterns")  # rules/summary_name_patterns.csv

if col_name_2 in content_df_ttl.columns:
    rows_before_2 = len(content_df_ttl)

    # Build regex pattern from list
    regex_pattern = '|'.join(re.escape(p) for p in target_patterns)

    # Case-insensitive contains check
    # Vectorized match
    mask_remove = (
        content_df_ttl[col_name_2]
        .fillna("")
        .astype(str)
        .str.contains(regex_pattern, case=False, na=False, regex=True)
    )

    n_removed_2 = mask_remove.sum()

    # print matched examples
    print(f"Examples of removed rows:")
    print(content_df_ttl.loc[mask_remove, col_name_2].drop_duplicates())

    content_df_ttl = content_df_ttl.loc[~mask_remove].copy()
    rows_after_2 = len(content_df_ttl)

    print(f"\nRemoved rows: {(n_removed_2):,} \ncontaining: \n'{target_patterns}'\n")   
    print(f"Rows before: {(rows_before_2):,}")
    print(f"Rows after: {(rows_after_2):,}\n")
else:
    print(f"\nColumn '{col_name_2}' not found. Skipping cleanup.\n")

print(f"Final row count: {len(content_df_ttl):,}\n")
print(f"Rows where source_type == 'Nutrient' & orig_name contains 'total' dropped | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Dropping rows where source_type == 'Nutrient' & orig_name contains 'total' | 2026-11-18 09:23:52

Initial row count: 5,145,532

Removed rows where source_type == 'Nutrient': 138,032
Rows before: 5,145,532
Rows after: 5,007,500

Examples of removed rows:
116684           Fatty acids, total saturated
116685    Fatty acids, total mono-unsaturated
116686    Fatty acids, total poly-unsaturated
116773                          Sugars, total
207530     Fatty acids, total monounsaturated
207531     Fatty acids, total polyunsaturated
Name: orig_name, dtype: str

Removed rows: 33,478 
containing: 
'['Carbohydrates, total', 'Carbohydrates, total available', 'Fat, total (Lipids)', 'Fatty acids, total monounsaturated', 'Fatty acids, total mono-unsaturated', 'Fatty acids, total polyunsaturated', 'Fatty acids, total poly-unsaturated', 'Fatty acids, total saturated', 'Sugars, total', 'Fiber, total dietary', 'Fiber, total dietary (AOAC)', 'Protein, total', 'Protein, total-N', 'Total lipid (fat)']'

Rows

## Step 8: Populate missing values in the Content table

- `orig_food_common_name` is filled within each `food_id`, then from the Food table.
- `food_name` is created from the Food table using `food_id`.
- `orig_name` is filled from the Compound table using `cpd_id`.
- Columns are reordered (`food_name` after `food_id`, `cpd_id` before `orig_name`).

In [18]:
# Populating content_df_ttl & cleanup
print(f"Populating food_name & orig_name columns missing values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_ppltd = content_df_ttl.copy()

print(f"Initial rows in content_df_ppltd: {len(content_df_ppltd):,}\n")
print(f"Initial columns: \n{content_df_ppltd.columns.tolist()}\n")

# fill orig_food_common_name using within-group values, then fallback to food_df_harm
col = 'orig_food_common_name'
key = 'food_id'

# baseline empty count (before any fill)
if col in content_df_ppltd.columns:
    empty_before = (
        content_df_ppltd[col].isna() |
        (content_df_ppltd[col].astype(str).str.strip() == '')
    ).sum()
    print(f"Empty '{col}' before filling: {empty_before:,}\n")
else:
    print(f"Missing column: {col}\n")

# within food_id fill (use any existing non-empty value)
if col in content_df_ppltd.columns and key in content_df_ppltd.columns:
    rows_before = len(content_df_ppltd)

    # normalize empties
    content_df_ppltd[col] = content_df_ppltd[col].replace(r'^\s*$', pd.NA, regex=True)

    # propagate first valid value within each food_id group
    # (groupby-level ffill/bfill avoids the object-dtype downcasting FutureWarning)
    content_df_ppltd[col] = content_df_ppltd.groupby(key)[col].ffill()
    content_df_ppltd[col] = content_df_ppltd.groupby(key)[col].bfill()

    print(f"Within-group fill completed")
else:
    print(f"Missing columns: {col} / {key}\n")

# fallback from food_df_harm.food_name
fallback_col = 'food_name'

if (col in content_df_ppltd.columns and
    key in content_df_ppltd.columns and
    fallback_col in food_df_harm.columns):
    rows_before = len(content_df_ppltd)

    # build lookup from food_df_harm
    food_lookup = (food_df_harm[[key, fallback_col]]
        .drop_duplicates(subset=key)
        .set_index(key)[fallback_col])

    # identify still-missing rows
    mask_missing = content_df_ppltd[col].isna() | (content_df_ppltd[col].astype(str).str.strip() == '')
    content_df_ppltd.loc[mask_missing, col] = content_df_ppltd.loc[mask_missing, key].map(food_lookup)
    n_filled = (~content_df_ppltd[col].isna()).sum()

    print(f"Fallback fill from food_df_harm completed")
else:
    print(f"Missing columns for fallback step")

# final empty count + filled summary
if col in content_df_ppltd.columns:
    empty_after = (content_df_ppltd[col].isna() |
        (content_df_ppltd[col].astype(str).str.strip() == '')).sum()

    filled_total = empty_before - empty_after

    print(f"Empty '{col}' after filling: {empty_after:,}")
    print(f"Total filled values: {filled_total:,}\n")
    

# populate/create food_name from food_df_harm using food_id
required_cols_content = ['food_id']
required_cols_food = ['food_id', 'food_name']

if (all(col in content_df_ppltd.columns for col in required_cols_content)
    and
    all(col in food_df_harm.columns for col in required_cols_food)):
    # Create lookup dictionary
    food_lookup = (food_df_harm[['food_id', 'food_name']]
        .drop_duplicates(subset='food_id')
        .set_index('food_id')['food_name'])

    # Populate/create food_name column
    content_df_ppltd['food_name'] = (content_df_ppltd['food_id'].map(food_lookup))
    n_food_names = content_df_ppltd['food_name'].notna().sum()

    print(f"food_name column created & populated")
    print(f"Matched food names: {n_food_names:,}\n")
else:
    print(f"Required columns missing for food_name population\n")


# populate empty orig_name from cpd_df_fltrd using cpd_id
required_cols_content = ['cpd_id', 'orig_name']
required_cols_cpd = ['cpd_id', 'orig_name']

if (all(col in content_df_ppltd.columns for col in required_cols_content)
    and
    all(col in cpd_df_fltrd.columns for col in required_cols_cpd)):

    rows_before = len(content_df_ppltd)

    # Create lookup dictionary
    cpd_lookup = (cpd_df_fltrd[['cpd_id', 'orig_name']]
        .drop_duplicates(subset='cpd_id')
        .set_index('cpd_id')['orig_name'])

    # Detect empty/missing orig_name values
    mask_empty = (content_df_ppltd['orig_name'].isna() |
        (content_df_ppltd['orig_name']
            .astype(str)
            .str.strip()
            .isin(['', 'nan', 'None', 'NA', 'N/A'])))

    n_empty_before = mask_empty.sum()

    # Populate only empty rows
    content_df_ppltd.loc[mask_empty, 'orig_name'] = (
        content_df_ppltd.loc[mask_empty, 'cpd_id']
        .map(cpd_lookup))

    # Recount remaining empty values
    mask_empty_after = (content_df_ppltd['orig_name'].isna() |
        (content_df_ppltd['orig_name']
            .astype(str)
            .str.strip()
            .isin(['', 'nan', 'None', 'NA', 'N/A'])))

    n_empty_after = mask_empty_after.sum()

    print(f"orig_name population complete")
    print(f"Empty values before: {n_empty_before:,}")
    print(f"Empty values after: {n_empty_after:,}")
    print(f"Filled values: {n_empty_before - n_empty_after:,}\n")
else:
    print(f"Required columns missing for orig_name population")
    

# Move food_name after food_id & cpd_id before orig_name
print(f"Initial columns before moving: \n{content_df_ppltd.columns.tolist()}\n")

required_cols = ['food_id', 'food_name', 'cpd_id', 'orig_name']

if all(col in content_df_ppltd.columns for col in required_cols):
    cols = content_df_ppltd.columns.tolist()

    # Remove food_name from current position
    cols.remove('food_name')
    # Insert after food_id
    idx_food_id = cols.index('food_id')
    cols.insert(idx_food_id + 1, 'food_name')
    
    # Remove cpd_id from current position
    cols.remove('cpd_id') 
    # Insert before orig_name
    idx_orig_name = cols.index('orig_name')
    cols.insert(idx_orig_name, 'cpd_id')

    # Reorder dataframe
    content_df_ppltd = content_df_ppltd[cols]

    print(f"food_name & cpd_id moved\n")
else:
    print(f"Required columns not found for reordering\n")

print(f"Final columns: \n{content_df_ppltd.columns.tolist()}\n")
print(f"Final row count: {len(content_df_ppltd):,}\n")

print(f"Done populating food_name & orig_name columns missing values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Populating food_name & orig_name columns missing values | 2026-11-18 09:23:56

Initial rows in content_df_ppltd: 4,974,022

Initial columns: 
['cont_id', 'cpd_id', 'food_id', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'orig_name', 'orig_content', 'orig_min', 'orig_max', 'orig_unit']

Empty 'orig_food_common_name' before filling: 970,137

Within-group fill completed
Fallback fill from food_df_harm completed
Empty 'orig_food_common_name' after filling: 0
Total filled values: 970,137

food_name column created & populated
Matched food names: 4,971,297

orig_name population complete
Empty values before: 4,426,588
Empty values after: 278,894
Filled values: 4,147,694

Initial columns before moving: 
['cont_id', 'cpd_id', 'food_id', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'orig_name', 'orig_content', 'orig_min', 'orig_max', 'orig_unit', 'food_name']

food_name & cpd_

## Step 9: Remove Content entries

Removes Content rows:

1. With no `food_name` or `orig_name` value.
2. With a `cpd_id` that is not present in the filtered Compound table.
3. With an empty or zero `orig_content` value.

In [19]:
# Further filtering of content_df_ppltd
print(f"Dropping rows with missing values in food_name, orig_name, orig_content & cpd_id | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_fltrd = content_df_ppltd.copy()

print(f"Initial row count: {len(content_df_fltrd):,}\n")

# drop rows with empty/missing food_name
col_name = 'food_name'

if col_name in content_df_fltrd.columns:
    rows_before = len(content_df_fltrd)

    mask_remove = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")


# drop rows with empty/missing orig_name
col_name = 'orig_name'

if col_name in content_df_fltrd.columns:

    rows_before = len(content_df_fltrd)

    mask_remove = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")


# drop rows with no matching cpd_id in cpd_df_fltrd
required_cols_content = ['cpd_id']
required_cols_cpd = ['cpd_id']

if (all(col in content_df_fltrd.columns for col in required_cols_content)
    and
    all(col in cpd_df_fltrd.columns for col in required_cols_cpd)):
    rows_before = len(content_df_fltrd)

    # Valid cpd_ids from cpd_df_fltrd
    valid_cpd_ids = set(cpd_df_fltrd['cpd_id'])

    # Keep only matching rows
    mask_keep = content_df_fltrd['cpd_id'].isin(valid_cpd_ids)
    n_removed = (~mask_keep).sum()
    content_df_fltrd = content_df_fltrd.loc[mask_keep].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed rows with unmatched cpd_id: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Required cpd_id columns not found\n")


# drop rows with empty/missing/zero orig_content
col_name = 'orig_content'

if col_name in content_df_fltrd.columns:
    rows_before = len(content_df_fltrd)

    # Empty/missing values
    mask_empty = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    # Numeric zero values
    mask_zero = (pd.to_numeric(content_df_fltrd[col_name], errors='coerce') == 0)

    # Print counts before removal
    n_empty = mask_empty.sum()
    n_zero = mask_zero.sum()

    print(f"Empty/missing '{col_name}' rows: {n_empty:,}")
    print(f"Zero-value '{col_name}' rows: {n_zero:,}\n")

    # Combined removal mask
    mask_remove = mask_empty | mask_zero
    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty/zero '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")
    

print(f"Final row count: {len(content_df_fltrd):,}\n")
print(f"Done dropping rows with missing values in food_name, orig_name, orig_content & cpd_id | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Dropping rows with missing values in food_name, orig_name, orig_content & cpd_id | 2026-11-18 09:24:00

Initial row count: 4,974,022

Removed empty 'food_name' rows: 2,725
Rows before: 4,974,022
Rows after: 4,971,297

Removed empty 'orig_name' rows: 278,397
Rows before: 4,971,297
Rows after: 4,692,900

Removed rows with unmatched cpd_id: 160,963
Rows before: 4,692,900
Rows after: 4,531,937

Empty/missing 'orig_content' rows: 4,082,352
Zero-value 'orig_content' rows: 180,011

Removed empty/zero 'orig_content' rows: 4,262,363
Rows before: 4,531,937
Rows after: 269,574

Final row count: 269,574

Done dropping rows with missing values in food_name, orig_name, orig_content & cpd_id | 2026-11-18 09:24:04



## Step 10: Descriptive statistics of the filtered tables

Shows stats (data types, unique values and missing values per column) for the filtered Food, Compound and Content tables.

In [20]:
# Show stats
print(f"Showing stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# df_stats is imported from inquire_common.utils in the configuration cell

# Run stats
df_stats(food_df_harm, "food_df_harm")
df_stats(cpd_df_fltrd, "cpd_df_fltrd")
df_stats(content_df_fltrd, "content_df_fltrd")

print(f"Done showing stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Showing stats | 2026-11-18 09:24:05

food_df_harm
Rows: 992
Columns: 12


,column,dtype,n_unique,n_missing,pct_missing
0,food_id,int64,992,0,0.00
1,food_name,str,992,0,0.00
11,public_food_id,str,992,0,0.00
3,description,str,939,47,4.74
2,food_name_scientific,str,662,317,31.96
5,legacy_id,float64,620,372,37.50
4,itis_id,float64,601,382,38.51
10,ncbi_taxonomy_id,float64,600,373,37.60
9,food_group_harmonized,str,142,0,0.00
7,food_subgroup,str,119,7,0.71



cpd_df_fltrd
Rows: 70,028
Columns: 16


,column,dtype,n_unique,n_missing,pct_missing
0,cpd_id,int64,70028,0,0.00
1,public_cpd_id,str,70028,0,0.00
2,orig_name,str,70028,0,0.00
7,moldb_smiles,str,69226,2,0.00
10,inchikey,str,69226,0,0.00
8,moldb_inchi,str,69226,0,0.00
15,InChIKey_PubChem,str,64200,5010,7.15
4,iupac_name_pubchem,str,64170,5039,7.20
3,name,str,64057,5160,7.37
11,iupac_name,str,44219,25232,36.03



content_df_fltrd
Rows: 269,574
Columns: 14


,column,dtype,n_unique,n_missing,pct_missing
0,cont_id,int64,269574,0,0.00
10,orig_content,float64,18398,0,0.00
4,orig_food_common_name,str,9082,0,0.00
3,orig_food_id,str,8563,16513,6.13
12,orig_max,float64,4616,246299,91.37
9,orig_name,str,3741,0,0.00
8,cpd_id,int64,3115,0,0.00
11,orig_min,float64,2907,246267,91.35
7,orig_source_id,str,1919,47712,17.70
1,food_id,int64,830,0,0.00


Done showing stats | 2026-11-18 09:24:05



## Step 11: Save the filtered tables

Writes the filtered Food, Compound and Content tables to CSV files in the output folder (`FOOD_FLTRD_CSV`, `COMPOUND_FLTRD_CSV` and `CONTENT_FLTRD_CSV` in the configuration cell).

In [21]:
# Save filtered DFs
print(f"Writing filtered DFs to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Save merged FooDB
food_df_harm.to_csv(FOOD_FLTRD_CSV, index=False)
print(f"Done writing filtered Food DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

cpd_df_fltrd.to_csv(COMPOUND_FLTRD_CSV, index=False)
print(f"Done writing filtered Compound DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

content_df_fltrd.to_csv(CONTENT_FLTRD_CSV, index=False)
print(f"Done writing filtered Content DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"All filtered DFs written to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Writing filtered DFs to CSV | 2026-11-18 09:24:05

Done writing filtered Food DF to CSV | 2026-11-18 09:24:05
Done writing filtered Compound DF to CSV | 2026-11-18 09:24:06
Done writing filtered Content DF to CSV | 2026-11-18 09:24:07
All filtered DFs written to CSV | 2026-11-18 09:24:07



## Step 12: Merge the filtered tables

Merges Content with Food on `food_id`, then with Compound on `cpd_id`.

- Overlapping columns are renamed with `_food`, `_cont` and `_cpd` suffixes, and the columns are reordered.
- `food_sub_id` (`<food_id>_<nn>`) is created by numbering the distinct `orig_food_common_name` values within each `food_id`.

In [22]:
# Merge all dfs
print(f"Merging filtered DFs into foodb_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename overlapping columns before merging
print(f"Renaming overlapping columns before merging | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

food_df_merge = food_df_harm.rename(columns={
    'food_name': 'food_name_food'
})

cpd_df_merge = cpd_df_fltrd.rename(columns={
    'orig_name': 'orig_name_cpd'
})

content_df_merge = content_df_fltrd.rename(columns={
    'food_name': 'food_name_cont',
    'orig_name': 'orig_name_cont'
})


# merge content_df with food_df using food_id
foodb_df_cont_food = content_df_merge.merge(
    food_df_merge,
    on='food_id',
    how='left'
)

print(f"Columns content_df + food_df: \n{foodb_df_cont_food.columns.tolist()}\n")

# merge resulting df with cpd_df using cpd_id
foodb_df = foodb_df_cont_food.merge(
    cpd_df_merge,
    on='cpd_id',
    how='left'
)

print(f"Columns after merge of all DBs: \n{foodb_df.columns.tolist()}\n")


# reorder columns (if columns exist)
print(f"Reordering columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

col_pairs = [
    ('food_id', 'food_name_food'),
    ('cpd_id', 'public_cpd_id'),
    ('inchikey', 'orig_name_cpd'),
    ('iupac_name', 'iupac_name_pubchem'),
    ('orig_food_common_name', 'food_name_scientific')
]

cols = foodb_df.columns.tolist()

for first_col, second_col in col_pairs:
    if first_col in cols and second_col in cols:
        # remove the first column from current position
        cols.remove(first_col)
        # recompute index of second column
        idx_second = cols.index(second_col)
        # insert first column immediately before second column
        cols.insert(idx_second, first_col)

        print(f"Moved '{first_col}' before '{second_col}'")
    else:
        print(f"Skipping: '{first_col}' or '{second_col}' not found\n")

# Apply new column order
foodb_df = foodb_df[cols]

print(f"\nColumn reordering complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print(f"Columns after reordering: \n{foodb_df.columns.tolist()}\n")


# create and populate food_sub_id
required_cols = ['food_id', 'orig_food_common_name']

if all(col in foodb_df.columns for col in required_cols):
    rows_before = len(foodb_df)

    # normalize names for grouping
    foodb_df['orig_food_common_name'] = (
        foodb_df['orig_food_common_name']
        .fillna('')
        .astype(str)
        .str.strip()
    )

    # build mapping within each food_id separately
    food_subid_map = {}

    for food_id, grp in foodb_df.groupby('food_id', sort=False):
        unique_names = (
            pd.Series(grp['orig_food_common_name'].unique())
            .loc[lambda x: x != '']
            .tolist()
        )

        # number from 1 within this food_id only
        for i, name in enumerate(unique_names, start=1):
            food_subid_map[(food_id, name)] = f"{food_id}_{i:02d}"

    # populate food_sub_id
    foodb_df['food_sub_id'] = [
        food_subid_map.get(
            (food_id, food_name),
            f"{food_id}_01"
        )
        for food_id, food_name in zip(
            foodb_df['food_id'],
            foodb_df['orig_food_common_name']
        )
    ]

    # Move food_sub_id after food_id
    cols = foodb_df.columns.tolist()

    if 'food_sub_id' in cols:
        cols.remove('food_sub_id')
        idx_food_id = cols.index('food_id')
        cols.insert(idx_food_id + 1, 'food_sub_id')
        foodb_df = foodb_df[cols]

    print(f"food_sub_id created and populated | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
else:
    print(f"Missing required columns: {required_cols}\n")

print(f"Final columns: \n{foodb_df.columns.tolist()}\n")
print(f"foodb_df shape: \n{foodb_df.shape[0]:,} rows × {foodb_df.shape[1]} columns\n")
print(f"Merged 3 filtered DFs into foodb_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Merging filtered DFs into foodb_df | 2026-11-18 09:24:07

Renaming overlapping columns before merging | 2026-11-18 09:24:07

Columns content_df + food_df: 
['cont_id', 'food_id', 'food_name_cont', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'cpd_id', 'orig_name_cont', 'orig_content', 'orig_min', 'orig_max', 'orig_unit', 'food_name_food', 'food_name_scientific', 'description', 'itis_id', 'legacy_id', 'food_group', 'food_subgroup', 'food_type', 'food_group_harmonized', 'ncbi_taxonomy_id', 'public_food_id']

Columns after merge of all DBs: 
['cont_id', 'food_id', 'food_name_cont', 'orig_food_id', 'orig_food_common_name', 'orig_food_scientific_name', 'orig_food_part', 'orig_source_id', 'cpd_id', 'orig_name_cont', 'orig_content', 'orig_min', 'orig_max', 'orig_unit', 'food_name_food', 'food_name_scientific', 'description_x', 'itis_id', 'legacy_id', 'food_group', 'food_subgroup', 'food_type', 'food_group_harmonized', 'ncbi_taxonomy

Show stats (data types, unique values and missing values per column) for the merged table.

In [23]:
# Show stats - Merged FooDB
print(f"Showing merged FooDB stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# df_stats is imported from inquire_common.utils in the configuration cell

# Run stats
df_stats(foodb_df, "Merged foodb_df", sort_by_unique=False)

print(f"Done showing merged FooDB stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Showing merged FooDB stats | 2026-11-18 09:24:08

Merged foodb_df
Rows: 269,574
Columns: 41


,column,dtype,n_unique,n_missing,pct_missing
0,cont_id,int64,269574,0,0.00
1,food_name_cont,str,830,0,0.00
2,orig_food_id,str,8563,16513,6.13
3,orig_food_scientific_name,str,222,261215,96.90
4,orig_food_part,str,55,260969,96.81
5,orig_source_id,str,1919,47712,17.70
6,orig_name_cont,str,3741,0,0.00
7,orig_content,float64,18398,0,0.00
8,orig_min,float64,2907,246267,91.35
9,orig_max,float64,4616,246299,91.37


Done showing merged FooDB stats | 2026-11-18 09:24:09



Save the merged table as a tab-separated file (`MERGED_CSV` in the configuration cell).

In [24]:
# Save merged FooDB
print(f"Writing merged FooDB to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Save merged FooDB as tab-separated CSV
foodb_df.to_csv(MERGED_CSV, index=False, sep="\t")
print(f"Merged FooDB written to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Writing merged FooDB to CSV | 2026-11-18 09:24:10
Merged FooDB written to CSV | 2026-11-18 09:24:17



## Step 13: Export per-compound statistics

Writes one row per `inchikey` from the merged table (`foodb_df`) to CSV: compound id, PubChem name, number of rows, number of distinct foods (`food_id`) and number of distinct harmonized food groups. Content values are not summarised because `orig_unit` differs between rows. The output path (`STATS_CSV`) is set in the configuration cell.

In [25]:
# Write stats DF to CSV file (path set in the configuration cell)
print(f"Writing stats DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

stats_df = (
    foodb_df
    .groupby("inchikey", sort=False)
    .agg(
        cpd_id=("cpd_id", "first"),
        name=("name", "first"),
        row_count=("cpd_id", "size"),
        unique_foods=("food_id", "nunique"),
        unique_food_groups=("food_group_harmonized", "nunique"),
    )
    .reset_index()
    .sort_values(["row_count", "inchikey"], ascending=[False, True], kind="stable")
)

stats_df.to_csv(STATS_CSV, index=False, encoding="utf-8")

print(f"Stats DF written to {STATS_CSV} ({len(stats_df):,} compounds) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Writing stats DF to CSV | 2026-11-18 09:24:18

Stats DF written to ./foodb_cleaned/FooDB_03_Compound_Food_Composition.csv (3,052 compounds) | 2026-11-18 09:24:18

